# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Brune Boukobza

**ID**: BWB78

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `~/Desktop/BEE 4750/hw5-bruneb`


In [2]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [17]:
##CODE TO PROBLEM 1.1

##Data from table
#Components
components = [
    "Food", "Paper", "Plastics", "Textiles", "Rubber", "Wood",
    "Yard", "Glass", "Ferrous", "Aluminum", "OtherMetal", "Misc"
]
#Percent total mass
mass_pct = [15, 40, 5, 3, 2, 5, 18, 4, 2, 2, 1, 3]
#Combustion ash
ash_pct = [8, 7, 5, 10, 15, 2, 2, 100, 100, 100, 100, 70]
#MRF recycling rate
recycle_pct = [0, 55, 15, 10, 0, 30, 40, 60, 75, 80, 50, 0]
#Convert to fractions
mass_frac = mass_pct ./ 100
ash_frac = ash_pct ./ 100
recycle_frac = recycle_pct ./ 100

#Recycling fraction
recycling = sum(mass_frac .* recycle_frac)
#Fraction of waste that actually becomes recycling residuals which is the material that goes to MRF but isn't successfully recycled
residual_frac = sum(mass_frac .* recycle_frac .* (1 .- recycle_frac))
#Ash fraction of raw MSW going directly to WTE
ash_raw_WTE = sum(mass_frac .* ash_frac)
#Ash fraction of recycling residuals
ash_residuals = sum((mass_frac .* recycle_frac .* (1 .- recycle_frac)) .* ash_frac) /
                sum(mass_frac .* recycle_frac .* (1 .- recycle_frac))

println("Recycling fraction = ", round(recycling, digits=3))
println("Ash fraction of raw MSW sent to WTE = ", round(ash_raw_WTE, digits=3))
println("Ash fraction of recycling residuals = ", round(ash_residuals, digits=3))


Recycling fraction = 0.378
Ash fraction of raw MSW sent to WTE = 0.164
Ash fraction of recycling residuals = 0.153


#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

a_cf: waste transported from city c to facility f. There are nine of these.
b_df: leftover waste being transported from disopsal facility d to f. There are 3 of these.
c_f: binary variable to see if facility f is working.

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

Minimize cost, so combine transportation and disposal. 
If land-fill=1, MRF=2, and WTE=3, then 

Transportation costs:
1.5(5a11+30a12+15a13+15a21+25a22+10Wa3+13a31+45a32+20a33+32b21+18b31+15b23)

Disposal costs:
LF 2000c1+50(a11+a21+a31+b21+b31)
MRF 1500c2+7(a12+a22+a32)+0.38(40)(a12+a22+a32)
WTE 2500c3+60(a13+a23+a33+b23)

Now:
2000c1+1500c2+2500c3+57.5a11+67.2a12+82.5a13+72.5a21+59.7a22+75a23+69.5a31+89.7a32+90a33+98b31+77b31+82.5b23

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

Constraints: mass balance, capacity, non-negativity, binary, and demand, where all waste from each city must be treated. 

Mass balance:
For ash mass balance, consider waste sent directly to WTE has an ash
fraction of 0.164 and waste sent from recycling to WTE has an ash content of 0.14.
0.164(a13+a23+a33)+0.14(b23) = b31

Recycling mass balance: b21+b23 = (1-0.38)(a12+a22+a32)

Capacity: 
LF: a11+a21+a31+b21+b31 <= 200
MRF: a12+a22+a32 <= 350
WTE: a13+a23+a33+b23 <= 210

Demand:
City one: a11+a12+a13 = 100
City two: a12+a22+a32 = 90
City three: a31+a32+a33 = 120

Binary: 
set c1 = 1 because the landfill always has to operate. Use m:
a12+a22+a32 <= m2*c2; m2 = 350
a13+a23+a33+b23 <= m3*c3, m3 = 210

Non-negativity defines all variables as being >=0

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [ ]:
##CODE FOR PROBLEM 1.5

m2 = 350
m3 = 210
waste = Model(HiGHS.Optimizer)
@variable(waste, a[1:9] >= 0)
@variable(waste, b[1:3] >= 0)

#Binary variables
@variable(waste, c[1:3], Bin)
@constraint(waste, commit2, a[2] + a[5] + a[8] <= m2 * c[2])
@constraint(waste, commit3, a[3] + a[6] + a[9] + b[3] <= m3 * c[3])

#Landfill is always operating constraint
@constraint(waste, commit1, c[1] == 1)
#Cost
@objective(waste, Min, 2000*c[1]+1500*c[2]+2500*c[3]+57.5*a[1]+67.2*a[2]+82.5*a[3]+72.5*a[4]
+59.7*a[5]+75*a[6]+69.5*a[7]+89.7*a[8]+90*a[9]+98*b[1]+77*b[2]+82.5*b[3])

#Capacity constraints
@constraint(waste, ca1, (a[1]+a[4]+a[7]+sum(b[1:2]) <= 200))
@constraint(waste, ca2, (a[2]+a[5]+a[8] <= 350))
@constraint(waste, ca3, (a[3]+a[6]+a[9]+b[3] <= 210))
#Waste demand constraints
@constraint(waste, d1, (sum(a[1:3]) == 100))
@constraint(waste, d2, (sum(a[4:6]) == 90))
@constraint(waste, d3, (sum(a[7:9]) == 120))
#Mass balance constraints
@constraint(waste, MRF, (0.62*(a[2]+a[5]+a[8]) == b[1]+b[3]))
@constraint(waste, WTE, (0.164*(a[3]+a[6]+a[9])+0.14*(b[3]) == b[2]))

optimize!(waste)

display(value.(a))
display(value.(b))
display(value.(c))
cost = objective_value.(waste)
display(cost)

9-element Vector{Float64}:
 100.0
   0.0
   0.0
  -0.0
  -0.0
  90.0
  78.42105263157896
   0.0
  41.578947368421034

3-element Vector{Float64}:
  0.0
 21.578947368421048
  0.0

3-element Vector{Float64}:
  1.0
 -0.0
  1.0

27853.94736842105

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 11 rows; 15 cols; 41 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 4e+02]
Presolving model
9 rows, 14 cols, 37 nonzeros  0s
8 rows, 13 cols, 35 nonzeros  0s
Presolve reductions: rows 8(-3); columns 13(-2); nonzeros 35(-6) 

Solving MIP model with:
   8 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   35 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective

The minimum objective is $27,793 per day.

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

It is best to not use the recycling facility because the transportation costs are high since the MRF facility is far from the three cities.

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

Attached to second pdf form

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

In [32]:
import Pkg
Pkg.add("CSV")
using CSV

#Load data
genfile = "data/generators.csv"
df = CSV.read(genfile, DataFrame)

plants = df.Plant
Pmin = Dict(df.Plant .=> df.Pmin)
Pmax = Dict(df.Plant .=> df.Pmax)
VarCost = Dict(df.Plant .=> df.VarCost)
Ramp = Dict(df.Plant .=> df.Ramp)

#Renewable generators
is_wind = df.Resource .== "Wind"
is_solar = df.Resource .== "Solar"

#Generator sets
thermal = plants[.!is_wind .& .!is_solar]
wind = plants[is_wind]
solar = plants[is_solar]

##Scenario Tree Data
#Period 1
d1 = 1100
cf_solar_t1 = 0.9
cf_wind_t1  = 0.45
#Period 2
scenarios = [
    (name="S1", d=1200, cf_s=0.95, cf_w=0.40, p=0.75*0.70),
    (name="S2", d=1200, cf_s=0.75, cf_w=0.50, p=0.75*0.30),
    (name="S3", d=1500, cf_s=0.95, cf_w=0.40, p=0.25*0.70),
    (name="S4", d=1500, cf_s=0.75, cf_w=0.50, p=0.25*0.30),
]
S = length(scenarios)
model = Model(HiGHS.Optimizer)
#Period 1 generation
@variables(model, begin
    g1[p in plants] >= 0
end)
#Period 2 generation
@variables(model, begin
    g2[p in plants, s in 1:S] >= 0
end)

##Constraints
#Period 1
for p in plants
    @constraint(model, Pmin[p] <= g1[p] <= Pmax[p])
end
#Renewables in period 1 have CF limits that we must account for
for p in wind
    @constraint(model, g1[p] <= cf_wind_t1 * Pmax[p])
end
for p in solar
    @constraint(model, g1[p] <= cf_solar_t1 * Pmax[p])
end
#Period 1 energy balance
@constraint(model, sum(g1[p] for p in plants) == d1)

#Period 2
for (s, sc) in enumerate(scenarios)
    #Generator limits
    for p in plants
        @constraint(model, Pmin[p] <= g2[p,s] <= Pmax[p])
    end
    #Renewable CF limits
    for p in wind
        @constraint(model, g2[p,s] <= sc.cf_w * Pmax[p])
    end
    for p in solar
        @constraint(model, g2[p,s] <= sc.cf_s * Pmax[p])
    end
    for p in plants
        @constraint(model, g2[p,s] - g1[p] <= Ramp[p])
        @constraint(model, g1[p] - g2[p,s] <= Ramp[p])
    end
    #Energy balance each scenario
    @constraint(model, sum(g2[p,s] for p in plants) == sc.d)
end


##Minimize Cost
@expression(model,
    cost1, sum(VarCost[p] * g1[p] for p in plants)
)
@expression(model,
    cost2,
    sum(scenarios[s].p * sum(VarCost[p] * g2[p,s] for p in plants)
        for s in 1:S)
)
@objective(model, Min, cost1 + cost2)

#Optimize
optimize!(model)

println("Optimal objective = ", objective_value(model))
println("\nPeriod 1 generation:")
for p in plants
    println(p, ": ", value(g1[p]))
end

println("\nPeriod 2 generation by scenario:")
for (s, sc) in enumerate(scenarios)
    println("\nScenario $(sc.name): p=$(sc.p)")
    for p in plants
        println("  ", p, ": ", value(g2[p,s]))
    end
end


   Resolving package versions...
  No Changes to `~/Desktop/BEE 4750/hw5-bruneb/Project.toml`
  No Changes to `~/Desktop/BEE 4750/hw5-bruneb/Manifest.toml`


Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
LP has 106 rows; 35 cols; 192 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [4e-01, 4e+02]
  Bound   [0e+00, 0e+00]
  RHS     [1e+02, 2e+03]
Presolving model
45 rows, 30 cols, 94 nonzeros  0s
9 rows, 20 cols, 28 nonzeros  0s
Dependent equations search running on 2 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
6 rows, 11 cols, 16 nonzeros  0s
Presolve reductions: rows 6(-100); columns 11(-24); nonzeros 16(-176) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     1.7720232374e+04 Pr: 2(1090) 0s
          2     1.7926750000e+04 Pr: 0(0) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterations: 2
Objective value     :  1.7926750000e+04
P-D obje

┌ Warning: thread = 1 warning: parsed expected 6 columns, but didn't reach end of line around data row: 3. Parsing extra columns and widening final columnset
└ @ CSV /Users/bruneboukobza/.julia/packages/CSV/XLcqT/src/file.jl:593
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 4. Filling remaining columns with `missing`
└ @ CSV /Users/bruneboukobza/.julia/packages/CSV/XLcqT/src/file.jl:592
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 5. Filling remaining columns with `missing`
└ @ CSV /Users/bruneboukobza/.julia/packages/CSV/XLcqT/src/file.jl:592
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 6. Filling remaining columns with `missing`
└ @ CSV /Users/bruneboukobza/.julia/packages/CSV/XLcqT/src/file.jl:592




Period 1 generation:
Biomass: 0.0
Hydroelectric: 195.0
Geothermal: 0.0
NG CCGT: 220.0
NG CT: 100.0
Wind: 135.0
Solar: 450.0

Period 2 generation by scenario:

Scenario S1: p=0.5249999999999999
  Biomass: 0.0
  Hydroelectric: 285.0
  Geothermal: 0.0
  NG CCGT: 220.0
  NG CT: 100.0
  Wind: 120.0
  Solar: 475.0

Scenario S2: p=0.22499999999999998
  Biomass: 0.0
  Hydroelectric: 355.0
  Geothermal: 0.0
  NG CCGT: 220.0
  NG CT: 100.0
  Wind: 150.0
  Solar: 375.0

Scenario S3: p=0.175
  Biomass: 85.0
  Hydroelectric: 500.0
  Geothermal: 0.0
  NG CCGT: 220.0
  NG CT: 100.0
  Wind: 120.0
  Solar: 475.0

Scenario S4: p=0.075
  Biomass: 100.0
  Hydroelectric: 500.0
  Geothermal: 0.0
  NG CCGT: 275.0
  NG CT: 100.0
  Wind: 150.0
  Solar: 375.0


## References

List any external references consulted, including classmates.

Holly Archer